In [1]:
# -*- coding: utf-8 -*-
"""
Tek script: (1) tarla bazlı iç içe (nested) CV ile out-of-fold tahminler,
            (2) R2 / RMSE / MAE tablosu (%95 GA), (3) eşleşik anlamlılık testleri, (4) şekiller.

NEDEN BU TASARIM?
 - Eski kod: KFold(shuffle) + Optuna AYNI katlarda hem ayarlıyor hem raporluyor, grup yok.
   -> iyimser R2 ve aynı tarlanın farklı yılları eğitim/doğrulamada birlikte (sızıntı).
 - Burada: dış katlar tarla (GROUP_COL) bazlı; Optuna YALNIZCA dış eğitim kısmının içinde (iç CV);
   her kayıt tam bir kez, hiç görmediği tarla modeliyle tahmin edilir (out-of-fold, OOF).
 - "En iyi algoritma" iç CV skoruyla seçilir (test/OOF skoruyla DEĞİL) -> seçim yanlılığı yok.

KULLANIM (Kaggle): dosyanın başındaki AYARLAR bölümünü doldurun, sonra
    python makale_analiz.py            # hepsini koşturur
Çıktılar OUT_DIR içine: oof_tahminler.csv, tablo4_metrikler.csv, kontrastlar.csv, sekil_*.png
"""
import os, sys, warnings, itertools
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import wilcoxon
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# =========================== AYARLAR ===========================
TARGET      = "rekolte_x"          # verim sütunu (kg/da ise UNIT_FACTOR=1)
UNIT_FACTOR = 1.0                  # kg/da -> kg/ha için 10 yazın
GROUP_COL   = "field_id"           # !!! TARLA kimliği sütunu (aynı tarlanın farklı yılları aynı değer).
                                   # CSV'de yoksa ekleyin (parsel kodu / koordinat). None ise normal KFold (UYARI verilir).
ID_COL      = "record_id"          # parsel-yıl kimliği; sensörler arası eşleştirme için (None ise satır sırası)
MODELS      = ["ExtraTrees", "RandomForest", "CatBoost", "GradientBoosting", "LightGBM"]
N_OUTER, N_INNER, N_TRIALS, SEED = 5, 3, 20, 42      # hızlı deneme için N_TRIALS=5
N_BOOT      = 2000
OUT_DIR     = "/kaggle/working/cikti"
DROP_COLS = ['Buday','ekim','HERBST Toplam','FUNGUST Toplam','dap','re','20-20-0','0.33','20-32-0',
             'region_name','date','weather_wind_speed','weather_wind_direction','weather_pressure_kpa',
             'weather_evaporation_mm','TVI','DVI','RVI','MSAVI','NDRE']

AY = ["Ara","Oca","Sub","Mar","Nis","May"]; AYNO = ["12","1","2","3","4","5"]
SUF = {"Landsat8":"Land", "Sentinel2":"sent", "Hibrit":"hib"}
CONFIG = {}   # (sensor, pencere) -> csv yolu.  Yolu olmayanlar None kalırsa atlanır.
for s, suf in SUF.items():
    #for a, n in zip(AY, AYNO):                                   # AYLIK: dosya adı kalıbını KENDİNİZE göre düzenleyin
    #    CONFIG[(s, f"Aylik_{a}")] = f"/kaggle/input/AYLIK/{suf}_{n}.csv"
    for p, (a, b) in zip(["12_1","1_2","2_3","3_4","4_5"], zip(AY[:-1], AY[1:])):   # İKİ AYLIK
        CONFIG[(s, f"Iki_{a}-{b}")] = f"/kaggle/input/datasets/aykutdurgut/2aylik/ikiAy{p}{suf}.csv"
# MEVSİMLİK / TÜM SEZON: bildirdiğiniz yollar (Landsat 8 ve eksikleri tamamlayın)
CONFIG[("Sentinel2","Mevsim_Kis")]   = "/kaggle/input/datasets/aykutdurgut/sentinel-spili/winter.csv"
CONFIG[("Sentinel2","Mevsim_Bahar")] = "/kaggle/input/datasets/aykutdurgut/sentinel-spili/bahar.csv"
CONFIG[("Sentinel2","Tum_Sezon")]    = "/kaggle/input/datasets/aykutdurgut/sentinel-spili/tumSezon.csv"
CONFIG[("Hibrit","Mevsim_Bahar")]    = "/kaggle/input/datasets/aykutdurgut/hibrist-spili/bahar.csv"
CONFIG[("Hibrit","Tum_Sezon")]       = "/kaggle/input/datasets/aykutdurgut/hibrist-spili/tumSezon.csv"
CONFIG[("Hibrit","Mevsim_Kis")]      = "/kaggle/input/datasets/aykutdurgut/hibrist-spili/winter.csv"          # varsa
CONFIG[("Landsat8","Mevsim_Kis")]    = "/kaggle/input/datasets/aykutdurgut/landsat-spili/winter.csv"   # <- yolu yazın
CONFIG[("Landsat8","Mevsim_Bahar")]  = "/kaggle/input/datasets/aykutdurgut/landsat-spili/bahar.csv"
CONFIG[("Landsat8","Tum_Sezon")]     = "/kaggle/input/datasets/aykutdurgut/landsat-spili/tumSezon.csv"

# Önceden belirlenmiş kontrastlar: (ad, (sensor, pencere) A, (sensor, pencere) B)   fark = A - B
CONTRASTS = [
 ("S2 Subat - S2 Tum sezon", ("Sentinel2","Aylik_Sub"), ("Sentinel2","Tum_Sezon")),
 ("L8 Mayis - L8 Tum sezon", ("Landsat8","Aylik_May"),  ("Landsat8","Tum_Sezon")),
 ("S2 - L8 (Subat)",         ("Sentinel2","Aylik_Sub"), ("Landsat8","Aylik_Sub")),
 ("L8 - Hibrit (Tum sezon)", ("Landsat8","Tum_Sezon"),  ("Hibrit","Tum_Sezon")),
 ("S2 - Hibrit (Tum sezon)", ("Sentinel2","Tum_Sezon"), ("Hibrit","Tum_Sezon")),
]
INTERACTION = (("Sentinel2","Aylik_Sub"),("Sentinel2","Tum_Sezon"),("Landsat8","Aylik_May"),("Landsat8","Tum_Sezon"))
# ================================================================

def make_model(trial, name):
    if name == "ExtraTrees":
        return ExtraTreesRegressor(n_estimators=trial.suggest_int("n_estimators", 50, 300),
            max_depth=trial.suggest_int("max_depth", 3, 20), min_samples_leaf=trial.suggest_int("msl", 1, 5),
            random_state=SEED, n_jobs=-1)
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=trial.suggest_int("n_estimators", 50, 300),
            max_depth=trial.suggest_int("max_depth", 3, 20), min_samples_leaf=trial.suggest_int("msl", 1, 5),
            random_state=SEED, n_jobs=-1)
    if name == "GradientBoosting":
        return GradientBoostingRegressor(n_estimators=trial.suggest_int("n_estimators", 50, 200),
            learning_rate=trial.suggest_float("lr", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6), random_state=SEED)
    if name == "LightGBM":
        return LGBMRegressor(n_estimators=trial.suggest_int("n_estimators", 50, 200),
            learning_rate=trial.suggest_float("lr", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 8), min_child_samples=trial.suggest_int("mcs", 3, 20),
            random_state=SEED, verbose=-1)
    if name == "CatBoost":
        return CatBoostRegressor(iterations=trial.suggest_int("iterations", 50, 200),
            learning_rate=trial.suggest_float("lr", 0.01, 0.3, log=True),
            depth=trial.suggest_int("depth", 3, 8), random_seed=SEED, verbose=0, thread_count=-1, train_dir='/tmp/catboost_info')
    raise ValueError(name)

def splitter(n_splits, groups, seed):
    if groups is None:
        return lambda X, y, g: KFold(n_splits, shuffle=True, random_state=seed).split(X)
    # gruplu + karıştırılmış: grup etiketlerini karıştırıp GroupKFold uygula
    def f(X, y, g):
        rng = np.random.default_rng(seed); u = np.unique(g); m = dict(zip(u, rng.permutation(len(u))))
        return GroupKFold(n_splits).split(X, y, np.array([m[v] for v in g]))
    return f

def load(path):
    df = pd.read_csv(path); df.columns = df.columns.str.strip()
    grp = df[GROUP_COL].values if (GROUP_COL and GROUP_COL in df.columns) else None
    rid = df[ID_COL].values if (ID_COL and ID_COL in df.columns) else np.arange(len(df))
    if GROUP_COL and grp is None: print(f"  ! UYARI: '{GROUP_COL}' yok -> gruplu bölme YAPILAMADI (normal KFold)")
    y = df[TARGET].values.astype(float) * UNIT_FACTOR
    X = df.drop(columns=[c for c in DROP_COLS + [TARGET, GROUP_COL, ID_COL] if c in df.columns], errors="ignore")
    X = X.select_dtypes(include=[np.number])
    return X.reset_index(drop=True), y, grp, rid

def nested_oof(X, y, grp, name):
    """Dış katlar: OOF tahmin. İç katlar: Optuna. Döner: oof (len(y),), ort. iç-CV R2"""
    oof = np.full(len(y), np.nan); inner_scores = []
    outer = splitter(N_OUTER, grp, SEED)
    for tr, te in outer(X, y, grp if grp is not None else None):
        Xtr, ytr = X.iloc[tr], y[tr]; gtr = grp[tr] if grp is not None else None
        inner = splitter(N_INNER, gtr, SEED + 1)
        def objective(trial):
            sc = []
            for a, b in inner(Xtr, ytr, gtr):
                m = make_model(trial, name).fit(Xtr.iloc[a], ytr[a])
                sc.append(r2_score(ytr[b], m.predict(Xtr.iloc[b])))
            return float(np.mean(sc))
        st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
        st.optimize(objective, n_trials=N_TRIALS)
        inner_scores.append(st.best_value)
        best = make_model(optuna.trial.FixedTrial(st.best_params), name).fit(Xtr, ytr)
        oof[te] = best.predict(X.iloc[te])
    return oof, float(np.mean(inner_scores))

def metrics(y, p):
    return dict(R2=r2_score(y, p), RMSE=float(np.sqrt(mean_squared_error(y, p))), MAE=mean_absolute_error(y, p))

# --------------------------- 1) OOF tahminleri ---------------------------
def run_oof():
    os.makedirs(OUT_DIR, exist_ok=True); rows = []
    for (sensor, win), path in CONFIG.items():
        if not path or not os.path.exists(path):
            print(f"- atlandı: {sensor} {win} (dosya yok/yol boş)"); continue
        print(f"\n== {sensor} | {win}"); X, y, grp, rid = load(path)
        res = {}
        for name in MODELS:
            oof, inner_r2 = nested_oof(X, y, grp, name); res[name] = (oof, inner_r2)
            print(f"   {name:17s} iç-CV R2={inner_r2:.3f} | OOF R2={r2_score(y, oof):.3f}")
        chosen = max(res, key=lambda k: res[k][1])               # SEÇİM: iç CV'ye göre (OOF'a değil)
        for name, (oof, inner_r2) in res.items():
            rows.append(pd.DataFrame(dict(sensor=sensor, window=win, model=name, chosen=(name == chosen),
                inner_cv_r2=inner_r2, record_id=rid, field_id=(grp if grp is not None else rid), y_true=y, y_pred=oof)))
        pd.concat(rows).to_csv(f"{OUT_DIR}/oof_tahminler.csv", index=False)   # ara kayıt
    return pd.concat(rows)

# --------------------------- 2) Metrik tablosu ---------------------------
def boot_idx(fields, B, rng):
    u = np.unique(fields); g = {f: np.where(fields == f)[0] for f in u}
    for _ in range(B):
        yield np.concatenate([g[f] for f in rng.choice(u, len(u), replace=True)])

def metric_table(df):
    rng = np.random.default_rng(SEED); out = []
    for (s, w), g in df[df.chosen].groupby(["sensor", "window"]):
        y, p, f = g.y_true.values, g.y_pred.values, g.field_id.values
        m = metrics(y, p)
        bs = np.array([[*metrics(y[i], p[i]).values()] for i in boot_idx(f, N_BOOT, rng)])
        lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
        out.append(dict(sensor=s, window=w, model=g.model.iloc[0], n=len(g),
            R2=m["R2"], R2_lo=lo[0], R2_hi=hi[0], RMSE=m["RMSE"], RMSE_lo=lo[1], RMSE_hi=hi[1],
            MAE=m["MAE"], MAE_lo=lo[2], MAE_hi=hi[2], rRMSE_pct=100*m["RMSE"]/y.mean()))
    t = pd.DataFrame(out).round(3); t.to_csv(f"{OUT_DIR}/tablo4_metrikler.csv", index=False)
    # Tüm algoritmalar (şeffaflık / medyan R2): 
    allm = df.groupby(["sensor","window","model"]).apply(
        lambda g: pd.Series(metrics(g.y_true.values, g.y_pred.values))).reset_index().round(3)
    allm.to_csv(f"{OUT_DIR}/tum_algoritmalar_metrikler.csv", index=False)
    return t

# --------------------------- 3) Anlamlılık testleri ---------------------------
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for i, j in enumerate(o): run = max(run, (m - i) * p[j]); adj[j] = min(1, run)
    return adj

def get(df, s, w):
    return df[df.chosen & (df.sensor == s) & (df.window == w)].drop_duplicates("record_id").set_index("record_id")

def boot_p(b): return min(1.0, 2 * min((b <= 0).mean(), (b >= 0).mean()))

def contrasts(df):
    rng = np.random.default_rng(SEED); out = []
    for name, a, b in CONTRASTS:
        A, B = get(df, *a), get(df, *b)
        if A.empty or B.empty: print("kontrast atlandı:", name); continue
        ids = A.index.intersection(B.index); A, B = A.loc[ids], B.loc[ids]
        y, pa, pb, f = A.y_true.values, A.y_pred.values, B.y_pred.values, A.field_id.values
        d = r2_score(y, pa) - r2_score(y, pb)
        bt = np.array([r2_score(y[i], pa[i]) - r2_score(y[i], pb[i]) for i in boot_idx(f, N_BOOT, rng)])
        dr = np.array([np.sqrt(mean_squared_error(y[i], pa[i])) - np.sqrt(mean_squared_error(y[i], pb[i]))
                       for i in boot_idx(f, N_BOOT, rng)])
        ea, eb = np.abs(y - pa), np.abs(y - pb)
        out.append(dict(kontrast=name, n_ortak=len(ids), dR2=d, dR2_lo=np.percentile(bt, 2.5), dR2_hi=np.percentile(bt, 97.5),
            p_bootstrap=boot_p(bt), dRMSE=np.sqrt(mean_squared_error(y, pa)) - np.sqrt(mean_squared_error(y, pb)),
            dRMSE_lo=np.percentile(dr, 2.5), dRMSE_hi=np.percentile(dr, 97.5),
            p_wilcoxon_mutlak_hata=(wilcoxon(ea, eb).pvalue if np.any(ea != eb) else 1.0)))
    # Etkileşim (fark-farkı)
    S1, S2, L1, L2 = [get(df, *k) for k in INTERACTION]
    if not any(x.empty for x in (S1, S2, L1, L2)):
        ids = S1.index
        for x in (S2, L1, L2): ids = ids.intersection(x.index)
        S1, S2, L1, L2 = [x.loc[ids] for x in (S1, S2, L1, L2)]; y = S1.y_true.values; f = S1.field_id.values
        fn = lambda i: ((r2_score(y[i], S1.y_pred.values[i]) - r2_score(y[i], S2.y_pred.values[i]))
                      - (r2_score(y[i], L1.y_pred.values[i]) - r2_score(y[i], L2.y_pred.values[i])))
        bt = np.array([fn(i) for i in boot_idx(f, N_BOOT, rng)])
        out.append(dict(kontrast="ETKILESIM (S2 Sub-Tum) - (L8 May-Tum)", n_ortak=len(ids), dR2=fn(np.arange(len(y))),
            dR2_lo=np.percentile(bt, 2.5), dR2_hi=np.percentile(bt, 97.5), p_bootstrap=boot_p(bt)))
    r = pd.DataFrame(out); r["p_bootstrap_Holm"] = holm(r.p_bootstrap.values)
    r.round(4).to_csv(f"{OUT_DIR}/kontrastlar.csv", index=False); return r

# --------------------------- 4) Şekiller ---------------------------
def figures(t):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    col = {"Landsat8": "#285d8c", "Sentinel2": "#1d6e43", "Hibrit": "#a05a1c"}
    groups = {"aylik": [w for w in t.window.unique() if w.startswith("Aylik")],
              "iki_aylik": [w for w in t.window.unique() if w.startswith("Iki")],
              "mevsim_tum": [w for w in t.window.unique() if w.startswith(("Mevsim", "Tum"))]}
    unit = "kg ha$^{-1}$" if UNIT_FACTOR == 10 else "kg da$^{-1}$"
    for gname, wins in groups.items():
        if not wins: continue
        fig, axs = plt.subplots(1, 3, figsize=(10.5, 3.5), dpi=150, sharey=True)
        for ax, s in zip(axs, ["Landsat8", "Sentinel2", "Hibrit"]):
            d = t[(t.sensor == s) & t.window.isin(wins)].set_index("window").reindex(wins).dropna(subset=["RMSE"])
            x = np.arange(len(d)); w = .38
            ax.bar(x - w/2, d.RMSE, w, color=col[s], edgecolor="black", lw=.6, label="RMSE")
            ax.bar(x + w/2, d.MAE, w, facecolor="white", edgecolor=col[s], hatch="////", label="MAE")
            for i, (a, b) in enumerate(zip(d.RMSE, d.MAE)):
                ax.text(i - w/2, a + 1, f"{a:.0f}", ha="center", fontsize=7, fontweight="bold"); ax.text(i + w/2, b + 1, f"{b:.0f}", ha="center", fontsize=7)
            ax.set_xticks(x); ax.set_xticklabels([f"{i.split('_',1)[1]}\n{m}" for i, m in zip(d.index, d.model)], fontsize=7)
            ax.set_title(s, loc="left", color=col[s], fontweight="bold"); ax.spines[["top", "right"]].set_visible(False)
            ax.legend(frameon=False, ncol=2, fontsize=7, loc="upper right")
        axs[0].set_ylabel(f"RMSE / MAE ({unit})"); axs[0].set_ylim(0, t[["RMSE"]].max().iloc[0] * 1.3)
        plt.tight_layout(); fig.savefig(f"{OUT_DIR}/sekil_hata_{gname}.png"); plt.close(fig)

if __name__ == "__main__":
    oof = run_oof()
    tab = metric_table(oof); print("\n", tab.to_string())
    print("\n", contrasts(oof).round(4).to_string())
    figures(tab); print("\nBitti ->", OUT_DIR)


== Landsat8 | Iki_Ara-Oca
  ! UYARI: 'field_id' yok -> gruplu bölme YAPILAMADI (normal KFold)
   ExtraTrees        iç-CV R2=0.649 | OOF R2=0.686
   RandomForest      iç-CV R2=0.612 | OOF R2=0.649
   CatBoost          iç-CV R2=0.631 | OOF R2=0.661
   GradientBoosting  iç-CV R2=0.604 | OOF R2=0.616
   LightGBM          iç-CV R2=0.620 | OOF R2=0.644

== Landsat8 | Iki_Oca-Sub
  ! UYARI: 'field_id' yok -> gruplu bölme YAPILAMADI (normal KFold)
   ExtraTrees        iç-CV R2=0.662 | OOF R2=0.689
   RandomForest      iç-CV R2=0.626 | OOF R2=0.669
   CatBoost          iç-CV R2=0.644 | OOF R2=0.667
   GradientBoosting  iç-CV R2=0.599 | OOF R2=0.642
   LightGBM          iç-CV R2=0.630 | OOF R2=0.649

== Landsat8 | Iki_Sub-Mar
  ! UYARI: 'field_id' yok -> gruplu bölme YAPILAMADI (normal KFold)
   ExtraTrees        iç-CV R2=0.646 | OOF R2=0.675
   RandomForest      iç-CV R2=0.623 | OOF R2=0.635
   CatBoost          iç-CV R2=0.636 | OOF R2=0.650
   GradientBoosting  iç-CV R2=0.614 | OOF R2=0.613
 